# Tutorial 2 — Semantic IDs: an identity built from content

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juanmigutierrez/generative-recommendation-engine/blob/main/notebooks/tutorial_02_semantic_ids.ipynb)

Companion to the *Semantic IDs* section. Here you will:

1. see what the Sentence-T5 embedding already knows about products (type any title fragment and get its neighbours),
2. see the anisotropy problem that silently collapsed the first RQ-VAE, and the fix,
3. **train the RQ-VAE yourself** (~1 min on GPU) with the project's JAX code and watch codebook utilisation and collisions,
4. look up any item's Semantic ID and browse everything that shares its first digit — the "category the model discovered on its own".

A GPU runtime is recommended (Runtime → Change runtime type → T4).

In [ ]:
#@title Setup — clone the repo, install deps, download the pre-computed artifacts (~1 min)
import os, sys, urllib.request
REPO_URL = "https://github.com/juanmigutierrez/generative-recommendation-engine"
RELEASE  = REPO_URL + "/releases/download/v1.0-artifacts"
QUICK    = os.environ.get("TUTORIAL_QUICK") == "1"   # tiny sizes for headless smoke tests

if not os.path.exists("backend"):
    if not os.path.exists("generative-recommendation-engine"):
        !git clone -q {REPO_URL}
    %cd generative-recommendation-engine
if "google.colab" in sys.modules:
    !pip install -q implicit lightgbm pyarrow ipywidgets 2>&1 | tail -1
os.makedirs("data/processed", exist_ok=True)

def fetch(*names):
    """Download an artifact from the GitHub release unless it is already on disk."""
    for n in names:
        p = os.path.join("data", "processed", n)
        if not os.path.exists(p):
            print("downloading", n, "...")
            urllib.request.urlretrieve(f"{RELEASE}/{n}", p)

fetch("item_catalog.parquet", "item_embeddings.npy", "item_embeddings_index.parquet", "semantic_ids.parquet")
for p in ["backend", "backend/scripts"]:
    if p not in sys.path: sys.path.insert(0, p)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets
pd.set_option("display.max_colwidth", 90)
P = os.path.join("data", "processed")
print("ready")

![The whole idea: each item is converted to a short sequence of discrete tokens (its Semantic ID); a generative model writes the next one; a lookup turns it back into an item.](https://raw.githubusercontent.com/juanmigutierrez/generative-recommendation-engine/main/blog/Images/genai_recommender.png)

*The whole idea: each item is converted to a short sequence of discrete tokens (its Semantic ID); a generative model writes the next one; a lookup turns it back into an item.*

## 1. Step 1 of the post: the title becomes 768 numbers

$$\mathbf{x} = \text{SentenceT5}(\text{title}) \in \mathbb{R}^{768}$$

The project embedded all 25,612 titles with `sentence-t5-base` (see `backend/scripts/build_item_embeddings_local.py`). The matrix is downloaded above; the optional cell after it re-embeds a few titles live so you can check the two agree.

In [ ]:
items = pd.read_parquet(f"{P}/item_catalog.parquet").set_index("item_id")
title = items["description"]
emb = np.load(f"{P}/item_embeddings.npy").astype("float32")
idx = pd.read_parquet(f"{P}/item_embeddings_index.parquet")
assert (idx["item_id"].values == np.arange(len(emb))).all()
print("embedding matrix:", emb.shape)
unit = emb / np.linalg.norm(emb, axis=1, keepdims=True)

def neighbours(query, n=6):
    """Find catalog titles containing `query`, take the first, list its nearest neighbours by cosine."""
    hits = title[title.str.contains(query, case=False, regex=False)]
    if len(hits) == 0: print("no title contains", repr(query)); return
    q = hits.index[0]
    sims = unit @ unit[q]; top = np.argsort(-sims)[1:n+1]
    print(f"query item: {title[q][:100]}\n")
    for j in top: print(f"  {sims[j]:.3f}  {title[j][:95]}")

interact(neighbours, query=widgets.Text(value="Jeecoo V20", description="title has"), n=widgets.IntSlider(6, 3, 12));

In [ ]:
#@title Optional — embed a few titles live with sentence-t5-base and compare (needs ~1 min to download the model)
RUN_LIVE_EMBED = False  #@param {type:"boolean"}
if RUN_LIVE_EMBED and not QUICK:
    !pip install -q sentence-transformers 2>&1 | tail -1
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("sentence-transformers/sentence-t5-base")
    sample = [0, 3, 100, 2000, 9000]
    live = model.encode([title[i] for i in sample], convert_to_numpy=True)
    live /= np.linalg.norm(live, axis=1, keepdims=True)
    for i, v in zip(sample, live):
        print(f"cosine(live, precomputed) = {float(v @ unit[i]):.4f}   {title[i][:70]}")

## 2. The trap: sentence embeddings are anisotropic

Every Sentence-T5 vector shares one big common direction. Two *random* products look 76% similar. Feed that straight into a VQ codebook initialised near the origin and every item snaps to the same code — the first RQ-VAE run in this project did exactly that. Standardising each dimension (subtract mean, divide by std) removes the shared direction; the project also seeds the codebooks with k-means on real encoder outputs.

In [ ]:
rng = np.random.RandomState(0)
a, b = rng.randint(0, len(emb), 5000), rng.randint(0, len(emb), 5000)
print(f"cosine similarity of RANDOM item pairs, raw embeddings:        {np.mean(np.sum(unit[a]*unit[b], axis=1)):.3f}")
emb_std = (emb - emb.mean(0)) / (emb.std(0) + 1e-8)
unit_std = emb_std / np.linalg.norm(emb_std, axis=1, keepdims=True)
print(f"cosine similarity of RANDOM item pairs, after standardising:   {np.mean(np.sum(unit_std[a]*unit_std[b], axis=1)):.3f}")

## 3. Steps 2–4: train the RQ-VAE

**Step 2: compress it.** A small encoder $E$ squeezes 768 numbers down to 32:

$$\mathbf{z} = E(\mathbf{x}), \qquad \mathbf{x} \in \mathbb{R}^{768}, \; \mathbf{z} \in \mathbb{R}^{32}$$

**Step 3: snap it to a codebook.** Instead of keeping $\mathbf{z}$ as 32 free numbers, snap it to the nearest of 256 learned reference vectors $C_1$; the first digit is the index of that vector. One snap is coarse, so take what's left over (the residual) and snap *that* to a second codebook, then a third:

$$c_1 = \arg\min_k \|\mathbf{z} - C_1[k]\|^2, \qquad \mathbf{r}_1 = \mathbf{z} - C_1[c_1]$$
$$c_2 = \arg\min_k \|\mathbf{r}_1 - C_2[k]\|^2, \qquad \mathbf{r}_2 = \mathbf{r}_1 - C_2[c_2], \qquad c_3 = \arg\min_k \|\mathbf{r}_2 - C_3[k]\|^2$$

The Semantic ID is $(c_1, c_2, c_3)$ — coarse, finer, finest — and the compressed vector is rebuilt by adding the three snapped vectors back together:

$$\hat{\mathbf{z}} = C_1[c_1] + C_2[c_2] + C_3[c_3]$$

Three codebooks of 256 give $256^3 \approx 16.7$ million possible IDs from only 768 learned vectors.

**Step 4: train it.** A decoder $D$ tries to rebuild the original 768 numbers from $\hat{\mathbf{z}}$. The loss makes that rebuild accurate while pulling the codebook vectors toward the data and the encoder toward the codebooks; $\mathrm{sg}[\cdot]$ is *stop-gradient* — the middle term moves only the codebooks, the last one only the encoder:

$$L = \|\mathbf{x} - D(\hat{\mathbf{z}})\|^2 + \|\mathrm{sg}[\mathbf{z}] - \hat{\mathbf{z}}\|^2 + \beta\,\|\mathbf{z} - \mathrm{sg}[\hat{\mathbf{z}}]\|^2, \qquad \beta = 0.25$$

![RQ-VAE: encode, quantise level by level on the residual, decode.](https://raw.githubusercontent.com/juanmigutierrez/generative-recommendation-engine/main/blog/Images/RQ_VAE.png)

*RQ-VAE: encode, quantise level by level on the residual, decode.*

That is what `backend/models/rqvae.py` implements: encoder 768 → 256 → 32, three codebooks of 256, decoder back to 768, straight-through estimator. The post's run used 400 epochs; 150 is enough to see the behaviour. On a T4 this is about a minute.

In [ ]:
import jax, jax.numpy as jnp, time
from models import rqvae
EPOCHS = 4 if QUICK else 150   #@param {type:"integer"}
HIDDEN, LATENT, LEVELS, CODES, BATCH, LR, BETA = 256, 32, 3, 256, 1024, 2e-3, 0.25

x = jnp.array(emb_std)
key = jax.random.PRNGKey(42); k_init, k_cb = jax.random.split(key)
params = rqvae.init_params(k_init, emb.shape[1], HIDDEN, LATENT, LEVELS, CODES)
params["codebooks"] = rqvae.kmeans_init_codebooks(k_cb, rqvae.encode(params, x), LEVELS, CODES)   # the k-means fix
opt_init, opt_update = rqvae.make_adam(lr=LR); opt_state = opt_init(params)
step = jax.jit(jax.value_and_grad(lambda p, b: rqvae.forward(p, b, beta=BETA)[0]))

n = len(emb); nb_ = n // BATCH; hist = []; t0 = time.time(); prng = np.random.RandomState(0)
for ep in range(EPOCHS):
    perm = prng.permutation(n); tot = 0.0
    for b in range(nb_):
        loss, g = step(params, x[perm[b*BATCH:(b+1)*BATCH]]); params, opt_state = opt_update(g, opt_state, params); tot += float(loss)
    hist.append(tot / nb_)
    if ep % 25 == 0 or ep == EPOCHS - 1: print(f"epoch {ep:3d}  loss {hist[-1]:.4f}  ({time.time()-t0:.0f}s)")

fig, ax = plt.subplots(figsize=(6, 3)); ax.plot(hist, color="#2a78d6"); ax.set_xlabel("epoch"); ax.set_ylabel("recon + codebook + β·commitment")
for s in ["top", "right"]: ax.spines[s].set_visible(False)
plt.show()

### What came out: utilisation and collisions

How many of the 256 codes does each level actually use, and how many items end up with an identical 3-digit code (and therefore need the 4th, tie-break digit)?

In [ ]:
from collections import Counter
codes = np.array(rqvae.get_codes(params, x))
for l in range(LEVELS):
    print(f"level {l+1}: {len(set(codes[:, l]))}/{CODES} codes used")
cnt = Counter(map(tuple, codes)); colliding = sum(c for c in cnt.values() if c > 1)
print(f"{len(cnt):,} unique 3-digit codes for {n:,} items; {colliding/n:.0%} of items share a code with another item → get a tie-break digit")

ref = pd.read_parquet(f"{P}/semantic_ids.parquet").set_index("item_id")
print(f"\n(the project's 400-epoch run: level-1 used {ref.sid_level_1.nunique()}/256, {(ref.groupby(['sid_level_1','sid_level_2','sid_level_3']).size()>1).sum():,} colliding codes)")

## 4. Browse the Semantic IDs

Type part of a title: you get the item's code from the model you just trained, and the other items that share its **first digit** — a family the model was never told about. Then narrow to items sharing the first *two* digits.

In [ ]:
def browse(query, shared_digits=1, n=12):
    hits = title[title.str.contains(query, case=False, regex=False)]
    if len(hits) == 0: print("no title contains", repr(query)); return
    q = hits.index[0]; c = codes[q]
    print(f"{title[q][:90]}\n→ Semantic ID (this run): {tuple(int(v) for v in c)}\n")
    mask = np.all(codes[:, :shared_digits] == c[:shared_digits], axis=1)
    fam = np.where(mask)[0]
    print(f"{len(fam):,} items share the first {shared_digits} digit(s). A sample:")
    for j in rng.choice(fam, min(n, len(fam)), replace=False): print(f"  {tuple(int(v) for v in codes[j])}  {title[j][:80]}")

interact(browse, query=widgets.Text(value="gaming keyboard", description="title has"),
         shared_digits=widgets.IntSlider(1, 1, 3, description="shared digits"), n=widgets.IntSlider(12, 5, 25));

## 5. The 2-D toy example from the post, in code

`z = (0.72, 0.31)`, codebook 1 = {(0.2, 0.8), (0.8, 0.2), (0.5, 0.5)}, codebook 2 = {(−0.1, 0.1), (0.1, −0.1), (0, 0)}.

In [ ]:
z = np.array([0.72, 0.31]); C1 = np.array([[0.2, 0.8], [0.8, 0.2], [0.5, 0.5]]); C2 = np.array([[-0.1, 0.1], [0.1, -0.1], [0.0, 0.0]])
c1 = int(np.argmin(((C1 - z) ** 2).sum(1))); r1 = z - C1[c1]
c2 = int(np.argmin(((C2 - r1) ** 2).sum(1))); z_hat = C1[c1] + C2[c2]
print(f"c1 = {c1} (nearest {C1[c1]}), residual {r1.round(2)}, c2 = {c2}, reconstruction {z_hat.round(2)} vs z {z} → error {(z - z_hat).round(2)}")

**Next:** [Tutorial 3 — Generative retrieval](https://colab.research.google.com/github/juanmigutierrez/generative-recommendation-engine/blob/main/notebooks/tutorial_03_generative_retrieval.ipynb): train a Transformer to *write* the next Semantic ID.